## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-11-09-47-59 +0000,nypost,Bernie Sanders won’t support Schumer for Senat...,https://nypost.com/2026/11/11/us-news/bernie-s...
1,2026-11-11-09-45-14 +0000,bbc,Tearful Smith switches allegiance to Netherlands,https://www.bbc.co.uk/sport/swimming/articles/...
2,2026-11-11-09-38-00 +0000,wsj,Vestas Wind Systems Shares Drop on Earnings Mi...,https://www.wsj.com/business/earnings/vestas-w...
3,2026-11-11-09-34-00 +0000,wsj,Trump’s Plan to Import Russian Diesel Stalls o...,https://www.wsj.com/business/energy-oil/trumps...
4,2026-11-11-09-33-00 +0000,wsj,Gold Rebounds But Fed Hike Expectations Cap Gains,https://www.wsj.com/finance/commodities-future...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2605/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
25,trump,29
163,nvidia,28
221,huang,23
291,new,21
220,jensen,18


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
127,2026-11-11-00-07-28 +0000,nypost,Chinese ‘spy’ Alice Huang met Nvidia CEO Jense...,https://nypost.com/2026/11/10/business/alice-h...,168
421,2026-11-10-11-59-50 +0000,nypost,FBI is ‘looking into’ Nvidia boss Jensen Huang...,https://nypost.com/2026/11/10/us-news/fbi-is-l...,151
271,2026-11-10-19-21-16 +0000,nypost,Nvidia loses nearly $450B in two-day wipeout a...,https://nypost.com/2026/11/10/business/nvidia-...,150
242,2026-11-10-20-22-14 +0000,nypost,Jensen Huang’s ties to suspected Chinese spy h...,https://nypost.com/2026/11/10/business/jensen-...,144
203,2026-11-10-21-35-09 +0000,nypost,Nvidia CEO Jensen Huang’s ‘spy’ affair could s...,https://nypost.com/2026/11/10/business/nvidia-...,135


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
127,168,2026-11-11-00-07-28 +0000,nypost,Chinese ‘spy’ Alice Huang met Nvidia CEO Jense...,https://nypost.com/2026/11/10/business/alice-h...
88,111,2026-11-11-02-15-00 +0000,wsj,House Republicans elected Rep. Jim Jordan of O...,https://www.wsj.com/politics/jim-jordan-wins-s...
318,104,2026-11-10-17-36-51 +0000,nypost,Gas prices drop below $4 a gallon for first ti...,https://nypost.com/2026/11/10/business/gas-pri...
118,80,2026-11-11-00-29-45 +0000,latimes,Meghan Markle's father Thomas Markle Sr. cut o...,https://www.latimes.com/entertainment-arts/sto...
114,76,2026-11-11-00-34-00 +0000,wsj,Defense Secretary Pete Hegseth gave a Pentagon...,https://www.wsj.com/politics/national-security...
383,59,2026-11-10-14-20-09 +0000,cbc,Anthropic eyes $1 trillion US valuation in hot...,https://www.cbc.ca/news/business/anthropic-ipo...
448,56,2026-11-10-10-48-00 +0000,nyt,Here’s why last year’s government shutdown wil...,https://www.nytimes.com/live/2026/11/10/busine...
102,53,2026-11-11-01-34-46 +0000,nypost,California Dems plotting sneaky mileage tax un...,https://nypost.com/2026/11/10/us-news/xavier-b...
244,51,2026-11-10-20-20-04 +0000,bbc,US Army soldier suspected of spying for Russia...,https://www.bbc.co.uk/news/articles/c6d7q77xgw...
72,49,2026-11-11-03-08-59 +0000,nypost,Family business of Ole Miss student who made m...,https://nypost.com/2026/11/10/us-news/family-b...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
